# Telco Customer Churn — EDA and Baseline

This notebook expects the IBM Telco Customer Churn CSV at `data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv`. It deliberately keeps customer identifiers out of model features.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from churn_prediction.data import load_telco_data

DATA_PATH = Path('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = load_telco_data(DATA_PATH)
df.head()

In [ ]:
print('Rows, columns:', df.shape)
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))
display(df['Churn'].value_counts().rename(index={0:'No', 1:'Yes'}).to_frame('customers'))

In [ ]:
churn_rate = df['Churn'].mean()
print(f'Churn rate: {churn_rate:.1%}')
sns.countplot(data=df, x='Churn')
plt.title('Target class distribution (0 = retained, 1 = churned)')
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
df[numeric_cols].hist(figsize=(12, 8), bins=25)
plt.tight_layout()
plt.show()

In [ ]:
for column in ['Contract', 'InternetService', 'PaymentMethod', 'PaperlessBilling']:
    if column in df.columns:
        rates = df.groupby(column, dropna=False)['Churn'].mean().sort_values(ascending=False)
        display((rates * 100).round(1).rename('churn_rate_pct').to_frame().style.format('{:.1f}%'))

## Baseline training

The baseline is trained from the CLI so that notebook exploration and model training remain reproducible. Run `python -m churn_prediction.train` from the repository root after installing the package and placing the dataset. Record the actual held-out metrics; the target ROC-AUC/recall are goals, not assumed results.